In [1]:
from unsloth import FastVisionModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

# 1. Load Model and Tokenizer
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit",
    load_in_4bit = True,
)

# 2. Add LoRA Adapters
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,

    r = 16, # Rank: higher = more parameters, lower = faster
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/manem/miniconda3/envs/qwen3-vl-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.339 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:29<00:00, 14.89s/it]


In [9]:
from datasets import load_dataset
from PIL import Image

# Load the local JSONL file
DATA_FILES = {
    "train": "/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_train_chatml.jsonl",
    "test": "/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test_chatml.jsonl"
}
dataset = load_dataset("json", data_files=DATA_FILES, split="train")

Generating train split: 510 examples [00:00, 34974.82 examples/s]
Generating test split: 90 examples [00:00, 39276.60 examples/s]


In [10]:
dataset

Dataset({
    features: ['messages'],
    num_rows: 510
})

In [11]:
def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = []
    for convo in convs:
        # We use tokenize=False to get the raw string for the SFTTrainer
        # add_generation_prompt=False because we want the assistant's answer included in the training text
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

# 2. Map the template to your dataset
# This creates the 'text' column that SFTTrainer looks for
dataset = dataset.map(formatting_prompts_func, batched=True)

Map: 100%|██████████| 510/510 [00:00<00:00, 4703.91 examples/s]


In [12]:
dataset

Dataset({
    features: ['messages', 'text'],
    num_rows: 510
})

In [13]:
dataset[2]

{'messages': [{'role': 'user',
   'content': [{'type': 'image',
     'image': '/home/manem/Qwen-3VL-Testing/dataset/images/train_images/f257.png',
     'text': None},
    {'type': 'text',
     'image': None,
     'text': 'Is there anything unrealistic in this image? yes or no or somewhat, if yes or somewhat explain in maximum 30 words, please ensure to explain what looks unreal like if face is distorted, or transition between objects is not smooth.Respond ONLY in the following JSON format:\n{\n  "unrealistic": "yes | no | somewhat",\n  "explanation": "string"\n}\n\n'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'image': None,
     'text': '{"unrealistic": "somewhat", "explanation": "The perspective and size of the chairs and tables seem slightly off, making them appear unnaturally small compared to the room\\u2019s size and height."}'}]}],
 'text': '<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|><|vision_start|><|image_pad|><|vision_end|><|im_end|>\n<|

In [14]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args = SFTConfig(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        output_dir="./qwen3-vl-8b-realm-finetuned",
        report_to="none",

        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048
    )
)

Unsloth: Model does not have a default image size - using 512


In [15]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 510 | Num Epochs = 2 | Total steps = 128
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,4.425400
2,4.472000
3,4.428900
4,4.236600
5,3.643100
6,3.016100
7,2.564000
8,2.190400
9,1.907400
10,1.693800


In [16]:
model.save_pretrained("qwen_lora_REALM_desc")
tokenizer.save_pretrained("qwen_lora_REALM_desc")

[]

In [ ]:
import os
from huggingface_hub import login

login(os.getenv("HF_TOKEN"))


model.push_to_hub("Machselo/qwen_lora_REALM_desc", token = HUGGING_FACE_TOKEN) 
tokenizer.push_to_hub("Machselo/qwen_lora_REALM_desc", token = HUGGING_FACE_TOKEN) 

Processing Files (1 / 1): 100%|██████████|  175MB /  175MB, 3.69MB/s  
New Data Upload: 100%|██████████|  175MB /  175MB, 3.69MB/s  


Saved model to https://huggingface.co/Machselo/qwen_lora_REALM_desc


Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB, 4.76MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
